# Tigress Bond Pricing Engine — BQuant Edition

Interactive new-issue corporate bond pricing tool, ported from the local `blpapi` version.

**What it does:** Type an equity ticker (e.g. `HUM`, `AAPL`, `JPM`), click **Load**. The notebook pulls peer-comparable bonds via BQL, fits a Nelson-Siegel-Svensson spread curve, and outputs indicative pricing across 2Y–30Y. Sliders shock spreads, rates, NIC, and deal size in real time (no BQL re-hit — cached after Load).

**Pipeline:** Issuer info → US Treasury curve (YCGT0025) → peer discovery (GICS → static map, or user override) → outstanding USD corp bonds for peer set (`bq.univ.bonds(..., issuedby='Credit_Family')`) → NSS regression on OAS spreads → per-tenor pricing chain (peer + credit + issuer + NIC) → sensitivity grid → notes.

**Run:** `Run All`. The dashboard renders inline at the bottom.

In [ ]:
# === Imports ===
import math

import numpy as np
import pandas as pd

import ipywidgets as widgets
import plotly.graph_objects as go

import bql

bq = bql.Service()


# === Constants (ported from local pricing.py) ===

TENORS = (2, 5, 7, 10, 20, 30)
TENOR_LABEL = {2: '2Y', 5: '5Y', 7: '7Y', 10: '10Y', 20: '20Y', 30: '30Y'}

RATING_SCALE = {
    'AAA': 1, 'AA+': 2, 'AA': 3, 'AA-': 4,
    'A+': 5, 'A': 6, 'A-': 7,
    'BBB+': 8, 'BBB': 9, 'BBB-': 10,
    'BB+': 11, 'BB': 12, 'BB-': 13,
    'B+': 14, 'B': 15, 'B-': 16,
    'CCC+': 17, 'CCC': 18, 'NR': 10,
}

PRESETS = {
    'base':   {'spread_shock': 0,   'rate_shock': 0,   'nic': 5},
    'stress': {'spread_shock': 75,  'rate_shock': -25, 'nic': 10},
    'rally':  {'spread_shock': -20, 'rate_shock': 25,  'nic': 3},
}

SENS_SPREAD_SHOCKS = (-50, -25, 0, 25, 50, 75, 100, 150)
SENS_RATE_SHOCKS = (-100, -50, -25, 0, 25, 50, 100)

# Sector → peer ticker map (GICS sub-industry keys). Used as fallback when
# the user doesn't supply an explicit peer list and Bloomberg peer field
# isn't queried.
SECTOR_PEERS = {
    'Managed Health Care':       ['UNH', 'ELV', 'CI', 'CNC', 'MOH', 'HUM'],
    'Health Care Services':      ['CVS', 'MCK', 'CAH', 'COR', 'DGX', 'UNH', 'ELV', 'CI'],
    'Health Care Facilities':    ['HCA', 'THC', 'UHS', 'CYH', 'SEM'],
    'Health Care Equipment':     ['MDT', 'ABT', 'SYK', 'BSX', 'BAX', 'BDX', 'ZBH', 'EW'],
    'Pharmaceuticals':           ['JNJ', 'PFE', 'MRK', 'LLY', 'BMY', 'ABBV', 'AMGN', 'GILD'],
    'Biotechnology':             ['AMGN', 'GILD', 'REGN', 'VRTX', 'BIIB'],
    'Life Sciences Tools & Services': ['TMO', 'DHR', 'A', 'IQV', 'MTD'],
    'Systems Software':          ['MSFT', 'ORCL', 'CRM', 'INTU', 'ADBE', 'NOW'],
    'Application Software':      ['ORCL', 'CRM', 'INTU', 'ADBE', 'SAP', 'WDAY', 'SNPS'],
    'Technology Hardware, Storage & Peripherals': ['AAPL', 'HPQ', 'HPE', 'DELL', 'WDC', 'STX'],
    'Semiconductors':            ['INTC', 'TXN', 'QCOM', 'AVGO', 'ADI', 'MCHP', 'NXPI'],
    'IT Consulting & Other Services': ['ACN', 'IBM', 'CTSH', 'INFY', 'WIT'],
    'Communications Equipment':  ['CSCO', 'MSI', 'JNPR', 'VIAV'],
    'Diversified Banks':         ['JPM', 'BAC', 'WFC', 'C', 'USB', 'PNC'],
    'Regional Banks':            ['USB', 'PNC', 'TFC', 'FITB', 'KEY', 'RF', 'HBAN', 'MTB', 'CFG'],
    'Investment Banking & Brokerage': ['GS', 'MS', 'SCHW', 'RJF', 'IBKR'],
    'Property & Casualty Insurance': ['AIG', 'TRV', 'ALL', 'CB', 'PGR', 'HIG'],
    'Life & Health Insurance':   ['MET', 'PRU', 'AFL', 'UNUM', 'LNC', 'GL'],
    'Consumer Finance':          ['AXP', 'COF', 'DFS', 'SYF', 'ALLY'],
    'Asset Management & Custody Banks': ['BLK', 'BK', 'STT', 'TROW', 'IVZ'],
    'Financial Exchanges & Data': ['ICE', 'CME', 'NDAQ', 'CBOE', 'SPGI', 'MCO'],
    'Packaged Foods & Meats':    ['GIS', 'K', 'CAG', 'SJM', 'CPB', 'HRL', 'MDLZ', 'HSY'],
    'Household Products':        ['PG', 'KMB', 'CLX', 'CL', 'CHD'],
    'Soft Drinks & Non-alcoholic Beverages': ['KO', 'PEP', 'MNST', 'KDP'],
    'Tobacco':                   ['PM', 'MO', 'BTI'],
    'Hypermarkets & Super Centers': ['WMT', 'COST', 'TGT'],
    'Restaurants':               ['MCD', 'SBUX', 'YUM', 'DPZ', 'CMG', 'QSR'],
    'Hotels, Resorts & Cruise Lines': ['MAR', 'HLT', 'H', 'RCL', 'CCL', 'NCLH'],
    'Home Improvement Retail':   ['HD', 'LOW'],
    'Broadline Retail':          ['AMZN', 'EBAY', 'TGT', 'COST'],
    'Automobile Manufacturers':  ['F', 'GM', 'TM', 'STLA'],
    'Aerospace & Defense':       ['BA', 'RTX', 'LMT', 'NOC', 'GD', 'HII', 'LHX', 'TDG'],
    'Industrial Conglomerates':  ['GE', 'HON', 'MMM', 'ITW', 'EMR'],
    'Railroads':                 ['UNP', 'CSX', 'NSC', 'CP'],
    'Airlines':                  ['DAL', 'UAL', 'LUV', 'AAL'],
    'Air Freight & Logistics':   ['FDX', 'UPS', 'EXPD', 'XPO'],
    'Integrated Oil & Gas':      ['XOM', 'CVX', 'COP', 'OXY', 'PSX', 'TTE'],
    'Oil & Gas Exploration & Production': ['COP', 'EOG', 'DVN', 'FANG', 'MRO', 'APA'],
    'Oil & Gas Refining & Marketing': ['PSX', 'VLO', 'MPC', 'PBF'],
    'Electric Utilities':        ['D', 'DUK', 'SO', 'NEE', 'AEP', 'EXC', 'SRE', 'XEL', 'ED', 'WEC'],
    'Multi-Utilities':           ['D', 'DUK', 'SO', 'SRE', 'ES', 'WEC', 'CMS', 'AEE'],
    'Integrated Telecommunication Services': ['T', 'VZ', 'TMUS'],
    'Cable & Satellite':         ['CMCSA', 'CHTR', 'DISH'],
    'Movies & Entertainment':    ['DIS', 'NFLX', 'WBD', 'PARA'],
    'Interactive Media & Services': ['GOOGL', 'META'],
    'Specialized REITs':         ['AMT', 'CCI', 'EQIX', 'PLD', 'DLR'],
    'Retail REITs':              ['SPG', 'O', 'NNN', 'KIM', 'REG', 'FRT'],
    'Industrial REITs':          ['PLD', 'DLR', 'EQIX'],
    'Specialty Chemicals':       ['APD', 'LIN', 'ECL', 'SHW', 'PPG', 'DD'],
    'Steel':                     ['NUE', 'STLD', 'CLF', 'X'],
}

# Hand-curated overrides for issuers whose CREDIT peers don't match their
# GICS sub-industry classification.
TICKER_PEER_OVERRIDE = {
    'AMZN': ['GOOGL', 'MSFT', 'ORCL', 'META', 'EBAY', 'TGT', 'COST'],
}


# === Pure pricing math (no BQL) =====================================

def _rating_to_num(r):
    return RATING_SCALE.get((r or 'NR').upper(), 10)


def compute_credit_adj(issuer_rating, peer_ratings):
    """Rating differential vs peer average × per-tenor bps multiplier.
    Multiplier scales from 3 bps/notch at 2Y to 8 bps/notch at 30Y.
    """
    peer_ratings = [r for r in peer_ratings if r]
    if not peer_ratings:
        return {t: 0 for t in TENORS}
    avg_peer = sum(_rating_to_num(r) for r in peer_ratings) / len(peer_ratings)
    issuer_num = _rating_to_num(issuer_rating)
    diff = issuer_num - avg_peer
    return {t: round(diff * (3 + (t / 30) * 5)) for t in TENORS}


def compute_issuer_adj(premium_bps):
    """Issuer premium × per-tenor scale: 0.57x at 2Y → 1.5x at 30Y."""
    return {t: round(premium_bps * (0.5 + (t / 30) * 1.0)) for t in TENORS}


def effective_nic(base_nic, deal_size_mm):
    """Big deals pay an execution premium."""
    bump = 3 if deal_size_mm > 2000 else (1 if deal_size_mm > 1500 else 0)
    return base_nic + bump, bump


def compute_pricing(treasuries, peer_curve_bps, peer_ratings, issuer_rating, scenario):
    credit_adj = compute_credit_adj(issuer_rating, peer_ratings)
    issuer_adj = compute_issuer_adj(scenario['issuer_premium'])
    eff_nic, nic_bump = effective_nic(scenario['nic'], scenario['deal_size'])
    rows = []
    for t in TENORS:
        tsy = treasuries.get(t, 0.0) + scenario['rate_shock'] / 100.0
        base_peer = max(0, round(peer_curve_bps.get(t, 0)))
        peer = base_peer + scenario['spread_shock']
        final_s = peer + credit_adj[t] + issuer_adj[t] + eff_nic
        yld = tsy + final_s / 100.0
        rows.append({
            'tenor':        t,
            'label':        TENOR_LABEL[t],
            'treasury':     round(tsy, 3),
            'peer':         peer,
            'credit':       credit_adj[t],
            'issuer':       issuer_adj[t],
            'nic':          eff_nic,
            'final_spread': final_s,
            'yield':        round(yld, 3),
            'ipt':          final_s + 12,
            'guidance':     final_s + 5,
        })
    return {
        'credit_adj':    credit_adj,
        'issuer_adj':    issuer_adj,
        'effective_nic': eff_nic,
        'nic_bump':      nic_bump,
        'rows':          rows,
    }


def compute_sensitivity(treasuries, peer_curve_bps, peer_ratings, issuer_rating, scenario):
    base = compute_pricing(
        treasuries, peer_curve_bps, peer_ratings, issuer_rating,
        {**scenario, 'spread_shock': 0, 'rate_shock': 0},
    )
    base_tsy = treasuries.get(10, 0.0)
    base_peer = max(0, round(peer_curve_bps.get(10, 0)))
    nic = base['effective_nic']
    credit_10 = base['credit_adj'][10]
    issuer_10 = base['issuer_adj'][10]
    grid = []
    for s in SENS_SPREAD_SHOCKS:
        row = []
        for r in SENS_RATE_SHOCKS:
            tsy = base_tsy + r / 100.0
            peer = base_peer + s
            final_s = peer + credit_10 + issuer_10 + nic
            row.append(round(tsy + final_s / 100.0, 3))
        grid.append(row)
    return {
        'spread_shocks':  list(SENS_SPREAD_SHOCKS),
        'rate_shocks':    list(SENS_RATE_SHOCKS),
        'grid':           grid,
        'current_spread': scenario['spread_shock'],
        'current_rate':   scenario['rate_shock'],
    }


def compute_notes(scenario, eff_nic, nic_bump):
    s = scenario['spread_shock']
    r = scenario['rate_shock']
    size = scenario['deal_size']
    out = []
    if s >= 50:
        out.append(('warn', f'Spread shock of +{s} bps implies a risk-off environment. Verify with desk head before circulating.'))
    if s <= -25:
        out.append(('info', f'Tight spread scenario ({s} bps). Check primary supply calendar — new issue may struggle for concession.'))
    if r >= 50:
        out.append(('warn', f'Treasury shock of +{r} bps raises absolute coupon cost significantly. Consider shorter tenors.'))
    if r <= -50:
        out.append(('ok', f'Rate rally of {r} bps. Potential refinancing window — flag to origination.'))
    if eff_nic > 10:
        out.append(('warn', f'Effective NIC of {eff_nic} bps is wide for IG (typical range 3-7 bps). Appropriate for stressed or debut issuers.'))
    if nic_bump > 0:
        out.append(('info', f'Deal size ${size/1000:.1f}B adds +{nic_bump} bps execution risk premium (effective NIC: {eff_nic} bps).'))
    out.append(('default', 'Output is indicative only. Final pricing subject to banker judgment and order book.'))
    return out


# === BQL data fetchers ==============================================

def fetch_issuer_info(ticker):
    """Pull name, rating, and sector for an equity ticker. Tries GICS sub-
    industry first (matches SECTOR_PEERS keys), then BICS L1 as fallback.
    """
    eq = f'{ticker.upper()} US Equity'
    items = {
        'name':       bq.data.name(),
        'rating':     bq.data.rating(),
        'gics':       bq.data.gics_sub_industry_name(),
        'bics':       bq.data.bics_level_1_sector_name(),
    }
    req = bql.Request(eq, items)
    try:
        resp = bq.execute(req)
    except Exception as e:
        return {'error': f'BQL request failed: {e}'}

    try:
        df = pd.concat([di.df()[di.name] for di in resp], axis=1)
    except Exception:
        return {'error': f'Ticker {ticker} returned no data.'}

    if df.empty:
        return {'error': f'Ticker {ticker} not found.'}

    row = df.iloc[0]
    name = row.get('name')
    if name is None or pd.isna(name):
        return {'error': f'Ticker {ticker} not found in Bloomberg.'}

    gics = row.get('gics')
    bics = row.get('bics')
    sector = (gics if gics and not pd.isna(gics) else
              bics if bics and not pd.isna(bics) else 'Unknown')

    return {
        'ticker': ticker.upper(),
        'name':   str(name),
        'rating': str(row.get('rating') or 'NR'),
        'sector': str(sector),
    }


def fetch_treasury_curve():
    """Pull US Treasury par curve via YCGT0025 Index (on-the-run UST)."""
    universe = bq.univ.curvemembers('YCGT0025 Index')
    items = {
        'tenor':    bq.data.id()['tenor'],
        'rate':     bq.data.curve_rate(side='mid'),
    }
    req = bql.Request(universe, items)
    resp = bq.execute(req)
    df = pd.concat([di.df()[di.name] for di in resp], axis=1)

    out = {}
    for _, row in df.iterrows():
        t_str = str(row.get('tenor', ''))
        # Parse '2Y', '5Y', etc.
        if t_str.endswith('Y'):
            try:
                t = int(t_str[:-1])
                rate = float(row['rate'])
                if t in TENORS:
                    out[t] = round(rate, 3)
            except (ValueError, TypeError):
                continue
    return out


def _peer_bond_universe(peer_tickers):
    """Shared filter chain for the peer-bond universe."""
    return (
        bq.univ.bonds(list(peer_tickers), issuedby='Credit_Family')
        .filter(
            (bq.data.amt_outstanding() >= '500M')
            .and_(bq.data.cpn_typ() == 'FIXED')
            .and_(bq.data.crncy() == 'USD')
            .and_(bq.data.maturity() > '6M')
            .and_(bq.data.cntry_of_risk() == 'US')
            .and_(bq.data.is_perpetual() == 'N')
        )
    )


def fetch_peer_bonds(peer_tickers):
    """Pull OAS-priced senior unsecured USD corp bonds for the peer set.
    Returns a DataFrame with rating, maturity, yield, spread, CDS basis,
    and within-rating spread Z-score.
    """
    if not peer_tickers:
        return pd.DataFrame()

    universe = _peer_bond_universe(peer_tickers)
    oas = bq.data.spread(spread_type='OAS')
    rating = bq.data.rating()

    items = {
        'name':         bq.data.name(),
        'issuer':       bq.data.issuer(),
        'ticker':       bq.data.ticker(),
        'rating':       rating,
        'duration':     bq.data.duration(fill='prev'),
        'maturity_yrs': bq.data.mty_years_tdy(),
        'yield':        bq.data.yield_(fill='prev'),
        'spread':       oas.round(0),
        'cds_basis':    bq.data.yas_zspread_basis_constant_mty().round(1),
        'zscore':       oas.groupzscore(by=rating).round(2),
    }
    req = bql.Request(universe, items, with_params={'fill': 'prev', 'mode': 'cached'})
    resp = bq.execute(req)
    df = pd.concat([di.df()[di.name] for di in resp], axis=1)
    df = df.dropna(subset=['spread', 'maturity_yrs'])
    df = df[(df['maturity_yrs'] >= 0.5) & (df['maturity_yrs'] <= 35) & (df['spread'] > 0)]
    return df.sort_values('maturity_yrs').reset_index(drop=True)


def fit_ns_curve(peer_tickers, target_tenors):
    """Fit Nelson-Siegel-Svensson via BQL `regression` on OAS spreads.
    Returns {tenor: predicted_spread_bps}.
    """
    if not peer_tickers:
        return {t: 0 for t in target_tenors}

    universe = _peer_bond_universe(peer_tickers)
    items = []
    for t in target_tenors:
        items.append(bq.data.regression(
            reg_by=bq.data.duration().group(),
            reg_value=bq.data.spread(spread_type='OAS').group(),
            regression_points=t,
            regression_type='nelson_siegel_svenson',
        ))
    req = bql.Request(universe, items)
    resp = bq.execute(req)

    out = {}
    for t, di in zip(target_tenors, resp):
        try:
            val = di.df()[di.name].iloc[0]
            out[t] = float(val) if not pd.isna(val) else 0
        except (IndexError, KeyError, TypeError, ValueError):
            out[t] = 0
    return out


def suggest_peers_from_sector(sector):
    """Static GICS sub-industry → peer ticker lookup with fuzzy fallback."""
    if not sector:
        return []
    if sector in SECTOR_PEERS:
        return list(SECTOR_PEERS[sector])
    for key, peers in SECTOR_PEERS.items():
        if sector.lower() in key.lower() or key.lower() in sector.lower():
            return list(peers)
    return []


# === Module-level state cache =======================================
# So slider moves don't re-hit BQL.

_state = {
    'ticker':         None,
    'issuer':         None,
    'treasuries':     {},
    'peer_bonds':     pd.DataFrame(),
    'peer_curve_bps': {},
    'peer_tickers':   [],
    'base_10y_spread': None,
}


def load_ticker(ticker, peer_override=None):
    """Hit BQL: refresh issuer, treasury, peer bonds, NS curve fit.
    Returns {'ok': True} or {'error': msg}.
    """
    ticker = ticker.upper().strip()
    issuer = fetch_issuer_info(ticker)
    if 'error' in issuer:
        return issuer

    try:
        treasuries = fetch_treasury_curve()
    except Exception as e:
        return {'error': f'Treasury curve fetch failed: {e}'}

    # Peer discovery
    if peer_override:
        peer_tickers = [p.strip().upper() for p in peer_override if p.strip()]
    elif ticker in TICKER_PEER_OVERRIDE:
        peer_tickers = list(TICKER_PEER_OVERRIDE[ticker])
    else:
        peer_tickers = suggest_peers_from_sector(issuer['sector'])

    peer_tickers = [p for p in peer_tickers if p != ticker]
    if not peer_tickers:
        return {'error': f'No peer mapping for sector "{issuer["sector"]}". Provide a peer list in the Override Peers box.'}

    try:
        peer_bonds = fetch_peer_bonds(peer_tickers)
    except Exception as e:
        return {'error': f'Peer bond fetch failed: {e}'}

    if peer_bonds.empty:
        return {'error': 'No peer bonds returned after filtering. Try a different peer set.'}

    try:
        peer_curve_bps = fit_ns_curve(peer_tickers, list(TENORS))
    except Exception as e:
        return {'error': f'NSS regression failed: {e}'}

    _state.update({
        'ticker':         ticker,
        'issuer':         issuer,
        'treasuries':     treasuries,
        'peer_bonds':     peer_bonds,
        'peer_curve_bps': peer_curve_bps,
        'peer_tickers':   peer_tickers,
        'base_10y_spread': None,  # reset on each load
    })
    return {'ok': True}


In [ ]:
# === CONSOLIDATED DASHBOARD (2026-06-10) ============================
# Everything in one block: BQL fetchers (with patches inline), widgets,
# charts, render, layout, initial load. No more v2/v3/v4/v5 patches.

# --- BQL helpers (overrides setup-cell with 'US Equity' fix inline) ---

def _peer_bond_universe(peer_tickers):
    """bq.univ.bonds with issuedby='Credit_Family' needs full equity IDs."""
    eq = [f'{t.upper().strip()} US Equity' for t in peer_tickers]
    return (
        bq.univ.bonds(eq, issuedby='Credit_Family')
        .filter(
            (bq.data.amt_outstanding() >= '500M')
            .and_(bq.data.cpn_typ() == 'FIXED')
            .and_(bq.data.crncy() == 'USD')
            .and_(bq.data.maturity() > '6M')
            .and_(bq.data.cntry_of_risk() == 'US')
            .and_(bq.data.is_perpetual() == 'N')
        )
    )

def _normalize_rating(val):
    if val is None: return 'NR'
    if isinstance(val, dict):
        return str(val.get('value') or val.get('VALUE') or 'NR')
    try:
        if pd.isna(val): return 'NR'
    except (TypeError, ValueError): pass
    return str(val)

def fetch_issuer_info(ticker):
    eq = f'{ticker.upper().strip()} US Equity'
    items = {
        'name':   bq.data.name(),
        'rating': bq.data.rating(),
        'gics':   bq.data.gics_sub_industry_name(),
        'bics':   bq.data.bics_level_1_sector_name(),
    }
    req = bql.Request(eq, items)
    try:
        resp = bq.execute(req)
        df = pd.concat([di.df()[di.name] for di in resp], axis=1)
    except Exception as e:
        return {'error': f'BQL issuer query failed: {e}'}
    if df.empty:
        return {'error': f'Ticker {ticker} not found.'}
    row = df.iloc[0]
    name = row.get('name')
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return {'error': f'Ticker {ticker} not found in Bloomberg.'}
    def _ok(v): return v is not None and not (isinstance(v, float) and pd.isna(v))
    gics, bics = row.get('gics'), row.get('bics')
    sector = gics if _ok(gics) else (bics if _ok(bics) else 'Unknown')
    return {
        'ticker': ticker.upper().strip(),
        'name':   str(name),
        'rating': _normalize_rating(row.get('rating')),
        'sector': str(sector),
    }

def fetch_peer_bonds(peer_tickers):
    """Pull peer bonds incl. CUSIP. Same shape used for issuer-own bonds."""
    if not peer_tickers:
        return pd.DataFrame()
    universe = _peer_bond_universe(peer_tickers)
    oas = bq.data.spread(spread_type='OAS')
    rating = bq.data.rating()
    items = {
        'name':         bq.data.name(),
        'issuer':       bq.data.issuer(),
        'ticker':       bq.data.ticker(),
        'cusip':        bq.data.id_cusip(),
        'rating':       rating,
        'duration':     bq.data.duration(fill='prev'),
        'maturity_yrs': bq.data.mty_years_tdy(),
        'issue_dt':     bq.data.issue_dt(),
        'maturity_dt':  bq.data.maturity(),
        'orig_tenor':   bq.data.maturity_years_at_issue(),
        'coupon':       bq.data.cpn(),
        'yield':        bq.data.yield_(fill='prev'),
        'spread':       oas.round(0),
        'cds_basis':    bq.data.yas_zspread_basis_constant_mty().round(1),
        'zscore':       oas.groupzscore(by=rating).round(2),
    }
    req = bql.Request(universe, items, with_params={'fill': 'prev', 'mode': 'cached'})
    resp = bq.execute(req)
    df = pd.concat([di.df()[di.name] for di in resp], axis=1)
    df = df.dropna(subset=['spread', 'maturity_yrs'])
    df = df[(df['maturity_yrs'] >= 0.5) & (df['maturity_yrs'] <= 35) & (df['spread'] > 0)]
    return df.sort_values('maturity_yrs').reset_index(drop=True)

def fetch_issuer_bonds(ticker):
    return fetch_peer_bonds([ticker])

def fetch_cusips(tickers):
    """Active USD bonds per ticker via bondsuniv('active') filter."""
    frames = []
    for tk in tickers:
        tk_clean = tk.upper().strip()
        try:
            universe = (
                bq.univ.bondsuniv('active', consolidateduplicates=False)
                .filter(bq.data.ticker() == tk_clean)
            )
            items = {
                'cusip':    bq.data.id_cusip(),
                'coupon':   bq.data.cpn(),
                'maturity': bq.data.maturity(),
                'currency': bq.data.crncy(),
                'name':     bq.data.name(),
            }
            req = bql.Request(universe, items)
            resp = bq.execute(req)
            df = pd.concat([di.df()[di.name] for di in resp], axis=1)
            if df.empty: continue
            df.insert(0, 'ticker', tk_clean)
            if 'currency' in df.columns:
                df = df[df['currency'].astype(str).str.upper() == 'USD']
            df = df.dropna(subset=['cusip'])
            frames.append(df)
        except Exception as e:
            print(f'CUSIP fetch failed for {tk_clean}: {e}')
            continue
    if not frames: return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    if 'maturity' in out.columns:
        out = out.sort_values(['ticker', 'maturity'])
    return out.reset_index(drop=True)

def load_ticker(ticker, peer_override=None):
    ticker = ticker.upper().strip()
    issuer = fetch_issuer_info(ticker)
    if 'error' in issuer:
        return issuer
    try:
        treasuries = fetch_treasury_curve()
    except Exception as e:
        return {'error': f'Treasury curve fetch failed: {e}'}
    if peer_override:
        peer_tickers = [p.strip().upper() for p in peer_override if p.strip()]
    elif ticker in TICKER_PEER_OVERRIDE:
        peer_tickers = list(TICKER_PEER_OVERRIDE[ticker])
    else:
        peer_tickers = suggest_peers_from_sector(issuer['sector'])
    peer_tickers = [p for p in peer_tickers if p != ticker]
    if not peer_tickers:
        return {'error': f'No peer mapping for sector "{issuer["sector"]}". Provide a peer list.'}
    try:
        peer_bonds = fetch_peer_bonds(peer_tickers)
    except Exception as e:
        return {'error': f'Peer bond fetch failed: {e}'}
    if peer_bonds.empty:
        return {'error': 'No peer bonds returned after filtering.'}
    try:
        peer_curve_bps = fit_ns_curve(peer_tickers, list(TENORS))
    except Exception as e:
        return {'error': f'NSS regression failed: {e}'}
    try:
        issuer_bonds = fetch_issuer_bonds(ticker)
    except Exception as e:
        print(f'Warning: issuer own-bond fetch failed for {ticker}: {e}')
        issuer_bonds = pd.DataFrame()
    try:
        cusips = fetch_cusips([ticker] + peer_tickers)
    except Exception as e:
        print(f'CUSIP fetch failed: {e}')
        cusips = pd.DataFrame()
    _state.update({
        'ticker': ticker, 'issuer': issuer, 'treasuries': treasuries,
        'peer_bonds': peer_bonds, 'peer_curve_bps': peer_curve_bps,
        'peer_tickers': peer_tickers, 'issuer_bonds': issuer_bonds,
        'cusips': cusips, 'base_10y_spread': None,
    })
    return {'ok': True}

# --- Color palette ---
BLUE_SHADES = ['#cfe2f3', '#a6c5e8', '#7fa9d6', '#5a8dc4',
               '#3672b2', '#1f5a9e', '#0d4485', '#062f6c']

def get_ticker_color_map(tickers):
    unique = sorted({t for t in tickers if t})
    return {t: BLUE_SHADES[i % len(BLUE_SHADES)] for i, t in enumerate(unique)}

# --- Widgets ---
ticker_input = widgets.Text(description='Issuer Ticker', value='HUM',
    placeholder='e.g. HUM, AAPL, JPM',
    style={'description_width': 'initial'}, layout={'width': '260px'})
peer_input = widgets.Text(description='Override Peers (optional)',
    placeholder='e.g. UNH, ELV, CI, CNC, MOH',
    style={'description_width': 'initial'}, layout={'width': '420px'})
load_button = widgets.Button(description='Load', button_style='success', layout={'width': '90px'})
issuer_label = widgets.HTML('<i style="color:#888;">Enter a ticker and click Load.</i>')
spinner = widgets.HTML('<i class="fa fa-spinner fa-spin" style="font-size:18px"></i>',
    layout={'visibility': 'hidden', 'margin': '3px 0 0 10px', 'width': '40px'})
exception_box = widgets.HTML(layout={'margin': '4px 0'})

deal_size_slider = widgets.IntSlider(min=250, max=3000, step=50, value=1000,
    description='Deal Size $M', style={'description_width': 'initial'},
    layout={'width': '360px'}, continuous_update=False)
spread_shock_slider = widgets.IntSlider(min=-50, max=150, step=5, value=0,
    description='IG Spread Shock (bps)', style={'description_width': 'initial'},
    layout={'width': '360px'}, continuous_update=False)
rate_shock_slider = widgets.IntSlider(min=-100, max=100, step=5, value=0,
    description='Treasury Shock (bps)', style={'description_width': 'initial'},
    layout={'width': '360px'}, continuous_update=False)
issuer_prem_slider = widgets.IntSlider(min=0, max=30, step=1, value=5,
    description='Issuer Premium (bps)', style={'description_width': 'initial'},
    layout={'width': '360px'}, continuous_update=False)
nic_slider = widgets.IntSlider(min=0, max=20, step=1, value=5,
    description='NIC (bps)', style={'description_width': 'initial'},
    layout={'width': '360px'}, continuous_update=False)

preset_base = widgets.Button(description='Base', layout={'width': '80px'})
preset_stress = widgets.Button(description='Stress', layout={'width': '80px'})
preset_rally = widgets.Button(description='Rally', layout={'width': '80px'})

def apply_preset(name):
    p = PRESETS[name]
    spread_shock_slider.value = p['spread_shock']
    rate_shock_slider.value = p['rate_shock']
    nic_slider.value = p['nic']

preset_base.on_click(lambda b: apply_preset('base'))
preset_stress.on_click(lambda b: apply_preset('stress'))
preset_rally.on_click(lambda b: apply_preset('rally'))

# KPI cards
kpi_10y_spread = widgets.HTML()
kpi_10y_yield  = widgets.HTML()
kpi_vs_base    = widgets.HTML()
kpi_peers      = widgets.HTML()

def kpi_card(label, value, sub=''):
    return (f'<div style="background:#fff;border-radius:8px;padding:14px 18px;'
            f'box-shadow:0 1px 3px rgba(0,0,0,0.08);min-width:170px;margin-right:10px;">'
            f'<div style="font-size:11px;color:#888;text-transform:uppercase;'
            f'letter-spacing:0.5px;font-weight:600;">{label}</div>'
            f'<div style="font-size:22px;font-weight:700;color:#1a1f36;'
            f'font-family:Consolas,monospace;margin-top:4px;">{value}</div>'
            f'<div style="font-size:11px;color:#888;margin-top:2px;">{sub}</div></div>')

# --- Charts ---
ns_chart = go.FigureWidget()
ns_chart.add_trace(go.Scatter(mode='markers', name='Peer bonds',
    marker=dict(size=9, opacity=0.75, line=dict(width=1, color='#0d4485'))))
ns_chart.add_trace(go.Scatter(mode='lines', name='NSS fitted curve',
    line=dict(color='#146ff8', width=2.5)))
ns_chart.add_trace(go.Scatter(mode='markers', name='Issuer indicative',
    marker=dict(size=14, symbol='diamond', color='#e8752a',
                line=dict(width=1.5, color='#c9621a'))))
ns_chart.add_trace(go.Scatter(mode='markers', name='Issuer own bonds',
    marker=dict(size=11, symbol='triangle-up', color='#10b981',
                line=dict(width=1.5, color='#047857')),
    hovertemplate='%{customdata}<br>Mat: %{x:.1f}y<br>Spread: %{y} bps<extra></extra>'))
ns_chart.update_layout(
    title='Peer spread curve (Nelson-Siegel-Svensson fit)',
    xaxis_title='Maturity (years)', yaxis_title='Spread (bps)',
    template='plotly_white', height=420, width=920,
    margin=dict(l=60, r=20, t=50, b=50),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)'),
)

coupon_vs_yield_chart = go.FigureWidget()
coupon_vs_yield_chart.add_trace(go.Bar(name='Coupon (at issue)', marker_color='#aab3c2',
    hovertemplate='%{x}<br>Coupon: %{y:.3f}%<extra></extra>'))
coupon_vs_yield_chart.add_trace(go.Bar(name='Current Yield', marker_color='#e8752a',
    hovertemplate='%{x}<br>Current yield: %{y:.3f}%<extra></extra>'))
coupon_vs_yield_chart.update_layout(
    title='Coupon (Issue Rate) vs Current Yield per peer bond',
    barmode='group', template='plotly_white', height=440, width=1100,
    margin=dict(l=60, r=20, t=50, b=140),
    yaxis_title='Yield (%)', xaxis_tickangle=-45,
    xaxis=dict(tickfont=dict(size=10)),
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.7)'),
)

waterfall_chart = go.FigureWidget()
waterfall_chart.add_trace(go.Waterfall(orientation='v',
    measure=['absolute','relative','relative','relative','relative','total'],
    x=['Treasury','Peer Sprd','Credit Adj','Issuer Adj','NIC','All-in Yield'],
    y=[0,0,0,0,0,0], text=['','','','','',''], textposition='outside',
    increasing={'marker':{'color':'#e8752a'}},
    decreasing={'marker':{'color':'#146ff8'}},
    totals={'marker':{'color':'#1a1f36'}}))
waterfall_chart.update_layout(title='10Y Spread Decomposition',
    template='plotly_white', height=380, width=550,
    margin=dict(l=40, r=20, t=50, b=40), yaxis_title='Yield / contribution (%)')

sens_chart = go.FigureWidget()
sens_chart.add_trace(go.Heatmap(z=[[0]], x=[0], y=[0],
    colorscale='RdYlGn_r', text=[[0]], texttemplate='%{text:.2f}',
    hovertemplate='Spread shock: %{y}bps<br>Rate shock: %{x}bps<br>10Y yield: %{z:.3f}%<extra></extra>',
    colorbar=dict(title='Yield %')))
sens_chart.update_layout(title='10Y Yield Sensitivity (%)',
    xaxis_title='Treasury Shock (bps)', yaxis_title='Spread Shock (bps)',
    template='plotly_white', height=380, width=550,
    margin=dict(l=70, r=20, t=50, b=50))

# --- HTML widgets ---
pricing_table       = widgets.HTML()
peer_table          = widgets.HTML()
issuer_bonds_table  = widgets.HTML()
issuer_bonds_header = widgets.HTML()
cusip_table         = widgets.HTML()
ns_color_legend     = widgets.HTML()
notes_box           = widgets.HTML()

# --- HTML builders ---

def _fmt(v, fmt, default='&mdash;'):
    if v is None: return default
    try:
        if pd.isna(v): return default
    except (TypeError, ValueError): pass
    try: return fmt.format(v)
    except (ValueError, TypeError): return default

def _fmt_date(v):
    if v is None: return '&mdash;'
    try:
        if pd.isna(v): return '&mdash;'
    except (TypeError, ValueError): pass
    try:
        return pd.Timestamp(v).strftime('%Y-%m-%d')
    except (TypeError, ValueError):
        s = str(v); return s[:10] if len(s) >= 10 else s

def pricing_table_html(pricing):
    rows = pricing['rows']
    html = '<table style="width:100%;border-collapse:collapse;font-size:13px;">'
    html += '<thead><tr style="background:#1a1f36;color:#c8ccd8;">'
    for h in ['Tenor','Treasury','Peer Sprd','Credit Adj','Issuer Adj','NIC','Final Spread','Yield','IPT','Guidance']:
        html += f'<th style="padding:10px 12px;text-align:right;font-weight:600;font-size:10px;text-transform:uppercase;letter-spacing:0.5px;">{h}</th>'
    html += '</tr></thead><tbody>'
    for r in rows:
        html += '<tr style="border-bottom:1px solid #eef0f3;">'
        html += f'<td style="padding:9px 12px;text-align:left;font-weight:700;">{r["label"]}</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["treasury"]:.3f}%</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["peer"]}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["credit"]:+d}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["issuer"]:+d}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["nic"]}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;font-weight:700;color:#146ff8;">{r["final_spread"]:.0f}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["yield"]:.3f}%</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["ipt"]:.0f}b</td>'
        html += f'<td style="padding:9px 12px;text-align:right;font-family:Consolas,monospace;">{r["guidance"]:.0f}b</td>'
        html += '</tr>'
    html += '</tbody></table>'
    return html

def bond_table_html(df):
    """Used for both peer and issuer-own bond tables. Includes CUSIP column."""
    if df.empty:
        return '<i style="color:#888;">No bonds.</i>'
    html = '<table style="width:100%;border-collapse:collapse;font-size:12px;">'
    html += '<thead><tr style="background:#1a1f36;color:#c8ccd8;">'
    headers = ['Issuer','Ticker','CUSIP','Rtg','Issued','Matures','Orig','Mat (Yrs)','Coupon','Yield','Spread','CDS Basis','Z']
    for h in headers:
        align = 'left' if h == 'Issuer' else 'right'
        html += (f'<th style="padding:7px 8px;text-align:{align};font-weight:600;'
                 f'font-size:10px;text-transform:uppercase;letter-spacing:0.3px;">{h}</th>')
    html += '</tr></thead><tbody>'
    for _, r in df.iterrows():
        issuer_str = str(r.get('issuer') or '')[:28]
        ticker_str = str(r.get('ticker') or '')
        cusip_str = str(r.get('cusip') or '&mdash;')
        rating_str = str(r.get('rating') or 'NR')
        z = r.get('zscore')
        if z is not None and not (isinstance(z, float) and pd.isna(z)):
            z_color = '#991b1b' if z > 1 else ('#166534' if z < -1 else '#1a1f36')
        else:
            z_color = '#888'
        html += '<tr style="border-bottom:1px solid #eef0f3;">'
        html += f'<td style="padding:5px 8px;text-align:left;">{issuer_str}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;font-weight:700;">{ticker_str}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;color:#555;">{cusip_str}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;">{rating_str}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;color:#666;">{_fmt_date(r.get("issue_dt"))}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;color:#666;">{_fmt_date(r.get("maturity_dt"))}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;">{_fmt(r.get("orig_tenor"), "{:.0f}y")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;">{_fmt(r.get("maturity_yrs"), "{:.1f}")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;color:#888;">{_fmt(r.get("coupon"), "{:.3f}%")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;">{_fmt(r.get("yield"), "{:.3f}%")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;font-weight:700;">{_fmt(r.get("spread"), "{:.0f}")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;">{_fmt(r.get("cds_basis"), "{:+.1f}")}</td>'
        html += f'<td style="padding:5px 8px;text-align:right;font-family:Consolas,monospace;color:{z_color};font-weight:700;">{_fmt(z, "{:+.2f}")}</td>'
        html += '</tr>'
    html += '</tbody></table>'
    return html

def cusip_table_html(df):
    if df.empty:
        return '<i style="color:#888;">No CUSIPs found.</i>'
    html = '<table style="width:100%;border-collapse:collapse;font-size:11px;">'
    html += '<thead><tr style="background:#1a1f36;color:#c8ccd8;">'
    for h in ['Ticker','CUSIP','Name','Coupon','Maturity','CCY']:
        align = 'left' if h in ('Ticker','CUSIP','Name') else 'right'
        html += (f'<th style="padding:6px 10px;text-align:{align};font-weight:600;'
                 f'font-size:10px;text-transform:uppercase;letter-spacing:0.3px;">{h}</th>')
    html += '</tr></thead><tbody>'
    for _, r in df.iterrows():
        cpn = r.get('coupon'); mty = r.get('maturity')
        try:
            cpn_str = f'{float(cpn):.3f}%' if cpn is not None and not pd.isna(cpn) else '&mdash;'
        except (TypeError, ValueError): cpn_str = '&mdash;'
        try:
            mty_str = pd.Timestamp(mty).strftime('%Y-%m-%d') if mty is not None and not pd.isna(mty) else '&mdash;'
        except (TypeError, ValueError): mty_str = '&mdash;'
        html += '<tr style="border-bottom:1px solid #eef0f3;">'
        html += f'<td style="padding:4px 10px;font-family:Consolas,monospace;font-weight:700;">{r.get("ticker") or ""}</td>'
        html += f'<td style="padding:4px 10px;font-family:Consolas,monospace;">{r.get("cusip") or "&mdash;"}</td>'
        html += f'<td style="padding:4px 10px;color:#555;">{str(r.get("name") or "")[:55]}</td>'
        html += f'<td style="padding:4px 10px;text-align:right;font-family:Consolas,monospace;">{cpn_str}</td>'
        html += f'<td style="padding:4px 10px;text-align:right;font-family:Consolas,monospace;color:#666;">{mty_str}</td>'
        html += f'<td style="padding:4px 10px;text-align:right;font-family:Consolas,monospace;">{r.get("currency") or "&mdash;"}</td>'
        html += '</tr>'
    html += '</tbody></table>'
    return html

def color_legend_html(color_map):
    if not color_map: return ''
    html = ('<div style="font-size:11px;color:#666;margin:4px 0 14px 0;'
            'padding:8px 12px;background:#fafafa;border-radius:4px;'
            'border-left:3px solid #146ff8;">')
    html += '<b style="color:#1a1f36;">Peer color key:</b> '
    for tk, col in color_map.items():
        html += (f'<span style="display:inline-block;margin:0 10px 0 4px;white-space:nowrap;">'
                 f'<span style="display:inline-block;width:11px;height:11px;background:{col};'
                 f'border-radius:50%;vertical-align:middle;margin-right:5px;border:1px solid #555;"></span>'
                 f'<span style="font-family:Consolas,monospace;font-weight:700;">{tk}</span></span>')
    html += '</div>'
    return html

def notes_html(notes):
    colors = {
        'warn':    ('#fef3cd','#856404','#f0ad4e'),
        'info':    ('#d1ecf1','#0c5460','#17a2b8'),
        'ok':      ('#d4edda','#155724','#28a745'),
        'default': ('#f0f4ff','#334155','#4a6cf7'),
    }
    html = '<div style="margin-top:6px;">'
    for kind, msg in notes:
        bg, fg, border = colors.get(kind, colors['default'])
        html += (f'<div style="padding:8px 12px;border-radius:6px;font-size:12px;'
                 f'margin-bottom:6px;line-height:1.5;background:{bg};color:{fg};'
                 f'border-left:3px solid {border};">{msg}</div>')
    html += '</div>'
    return html

# --- Render: pure-Python recompute from cached BQL data on every slider change ---

def get_scenario():
    return {
        'spread_shock':   spread_shock_slider.value,
        'rate_shock':     rate_shock_slider.value,
        'nic':            nic_slider.value,
        'issuer_premium': issuer_prem_slider.value,
        'deal_size':      deal_size_slider.value,
    }

def render(*args, **kwargs):
    if _state['ticker'] is None: return
    issuer = _state['issuer']
    treasuries = _state['treasuries']
    peer_bonds = _state['peer_bonds']
    peer_curve_bps = _state['peer_curve_bps']
    iss_bonds = _state.get('issuer_bonds', pd.DataFrame())
    cusips = _state.get('cusips', pd.DataFrame())

    scenario = get_scenario()
    peer_ratings = (peer_bonds['rating'].dropna().astype(str).tolist()
                    if 'rating' in peer_bonds.columns else [])

    pricing = compute_pricing(treasuries, peer_curve_bps, peer_ratings, issuer['rating'], scenario)
    sens    = compute_sensitivity(treasuries, peer_curve_bps, peer_ratings, issuer['rating'], scenario)
    notes   = compute_notes(scenario, pricing['effective_nic'], pricing['nic_bump'])

    r10 = next((r for r in pricing['rows'] if r['tenor'] == 10), None)
    if _state.get('base_10y_spread') is None and r10 is not None:
        base_p = compute_pricing(treasuries, peer_curve_bps, peer_ratings, issuer['rating'],
            {**scenario, 'spread_shock': 0, 'rate_shock': 0, 'nic': 5})
        b10 = next((x for x in base_p['rows'] if x['tenor'] == 10), None)
        _state['base_10y_spread'] = b10['final_spread'] if b10 else 0
    base_sprd = _state.get('base_10y_spread') or 0
    vs_base = (r10['final_spread'] - base_sprd) if r10 else 0

    # KPIs
    kpi_10y_spread.value = kpi_card('10Y Indicative',
        f'{r10["final_spread"]:.0f} bps' if r10 else '--', 'Final spread to Treasury')
    kpi_10y_yield.value = kpi_card('10Y Yield',
        f'{r10["yield"]:.3f}%' if r10 else '--', 'All-in indicative yield')
    kpi_vs_base.value = kpi_card('vs. Base', f'{vs_base:+.0f} bps', '10Y spread change')
    kpi_peers.value = kpi_card('Peers', f'{len(peer_bonds)} bonds', str(issuer['sector'])[:32])

    # NS chart: peer dots colored by issuer
    if not peer_bonds.empty:
        tickers = peer_bonds['ticker'].fillna('?').astype(str).tolist()
        color_map = get_ticker_color_map(tickers)
        colors = [color_map.get(t, '#1f5a9e') for t in tickers]
        peer_x = peer_bonds['maturity_yrs'].tolist()
        peer_y = peer_bonds['spread'].tolist()
        peer_custom = []
        for _, r in peer_bonds.iterrows():
            cpn = r.get('coupon')
            try:
                cpn_str = f'{float(cpn):.3f}%' if cpn is not None and not pd.isna(cpn) else '?'
            except (TypeError, ValueError): cpn_str = '?'
            peer_custom.append(f"{r.get('ticker') or '?'} {cpn_str}")
    else:
        color_map = {}; colors = []; peer_x, peer_y, peer_custom = [], [], []

    coarse_x = sorted(peer_curve_bps.keys())
    coarse_y = [peer_curve_bps[t] for t in coarse_x]
    fine_x = list(range(1, 31))
    fit_y = np.interp(fine_x, coarse_x, coarse_y).tolist() if coarse_x else []

    ind_x = [r['tenor'] for r in pricing['rows']]
    ind_y = [r['final_spread'] for r in pricing['rows']]

    if iss_bonds is not None and not iss_bonds.empty:
        own_x = iss_bonds['maturity_yrs'].tolist()
        own_y = iss_bonds['spread'].tolist()
        own_custom = []
        for _, r in iss_bonds.iterrows():
            cpn = r.get('coupon')
            try:
                cpn_str = f'{float(cpn):.3f}%' if cpn is not None and not pd.isna(cpn) else '?'
            except (TypeError, ValueError): cpn_str = '?'
            mty = r.get('maturity_dt')
            try:
                mty_str = pd.Timestamp(mty).strftime('%Y-%m-%d') if mty is not None and not pd.isna(mty) else '?'
            except (TypeError, ValueError): mty_str = '?'
            own_custom.append(f'{_state["ticker"]} {cpn_str} {mty_str}')
        own_name = f'{_state["ticker"]} own bonds ({len(iss_bonds)})'
    else:
        own_x, own_y, own_custom = [], [], []
        own_name = f'{_state["ticker"]} own bonds (none)'

    with ns_chart.batch_update():
        ns_chart.data[0].x = peer_x
        ns_chart.data[0].y = peer_y
        ns_chart.data[0].marker.color = colors
        ns_chart.data[0].customdata = peer_custom
        ns_chart.data[0].hovertemplate = '%{customdata}<br>Mat: %{x:.1f}y<br>Spread: %{y} bps<extra></extra>'
        ns_chart.data[1].x = fine_x
        ns_chart.data[1].y = fit_y
        ns_chart.data[2].x = ind_x
        ns_chart.data[2].y = ind_y
        ns_chart.data[3].x = own_x
        ns_chart.data[3].y = own_y
        ns_chart.data[3].customdata = own_custom
        ns_chart.data[3].name = own_name

    ns_color_legend.value = color_legend_html(color_map)
    pricing_table.value = pricing_table_html(pricing)

    if r10:
        tsy_yld = r10['treasury']
        wf_y = [tsy_yld, r10['peer']/100.0, r10['credit']/100.0, r10['issuer']/100.0, r10['nic']/100.0, r10['yield']]
        wf_text = [f'{tsy_yld:.3f}%', f'+{r10["peer"]}b', f'{r10["credit"]:+d}b', f'{r10["issuer"]:+d}b', f'+{r10["nic"]}b', f'{r10["yield"]:.3f}%']
        with waterfall_chart.batch_update():
            waterfall_chart.data[0].y = wf_y
            waterfall_chart.data[0].text = wf_text

    with sens_chart.batch_update():
        sens_chart.data[0].z = sens['grid']
        sens_chart.data[0].x = sens['rate_shocks']
        sens_chart.data[0].y = sens['spread_shocks']
        sens_chart.data[0].text = sens['grid']

    if not peer_bonds.empty:
        labels, coupons, yields = [], [], []
        for _, r in peer_bonds.iterrows():
            tk = str(r.get('ticker') or '')
            cpn = r.get('coupon'); mty = r.get('maturity_dt')
            try:
                mty_str = pd.Timestamp(mty).strftime('%m/%y') if mty is not None and not pd.isna(mty) else '?'
            except (TypeError, ValueError): mty_str = '?'
            try:
                cpn_str = f'{float(cpn):.2f}' if cpn is not None and not pd.isna(cpn) else '?'
            except (TypeError, ValueError): cpn_str = '?'
            labels.append(f'{tk} {cpn_str} {mty_str}')
            coupons.append(float(cpn) if cpn is not None and not (isinstance(cpn, float) and pd.isna(cpn)) else 0)
            yld = r.get('yield')
            yields.append(float(yld) if yld is not None and not (isinstance(yld, float) and pd.isna(yld)) else 0)
        with coupon_vs_yield_chart.batch_update():
            coupon_vs_yield_chart.data[0].x = labels
            coupon_vs_yield_chart.data[0].y = coupons
            coupon_vs_yield_chart.data[1].x = labels
            coupon_vs_yield_chart.data[1].y = yields

    peer_table.value = bond_table_html(peer_bonds)

    tk = _state['ticker']
    if iss_bonds is not None and not iss_bonds.empty:
        min_mty = iss_bonds['maturity_yrs'].min()
        max_mty = iss_bonds['maturity_yrs'].max()
        issuer_bonds_header.value = (
            f'<h4 style="margin:14px 0 8px 0;color:#1a1f36;">'
            f'{tk} Own Bonds &mdash; {len(iss_bonds)} active issues</h4>'
            f'<div style="font-size:11px;color:#888;margin-bottom:6px;">'
            f'Currently outstanding USD bonds for {tk}, ranging from '
            f'{min_mty:.1f}y to {max_mty:.1f}y remaining. '
            f'These are the GREEN triangles on the spread curve chart above.</div>'
        )
        issuer_bonds_table.value = bond_table_html(iss_bonds)
    else:
        issuer_bonds_header.value = (
            f'<h4 style="margin:14px 0 8px 0;color:#1a1f36;">{tk} Own Bonds</h4>'
            f'<div style="font-size:11px;color:#888;">No outstanding USD bonds found.</div>'
        )
        issuer_bonds_table.value = '<i style="color:#888;">No outstanding bonds.</i>'

    cusip_table.value = cusip_table_html(cusips)
    notes_box.value = notes_html(notes)


def on_load_click(b=None):
    spinner.layout.visibility = 'visible'
    exception_box.value = ''
    try:
        ticker = ticker_input.value.strip().upper()
        if not ticker:
            exception_box.value = '<span style="color:#c0392b;">Enter a ticker first.</span>'
            return
        peer_str = peer_input.value.strip()
        peer_override = ([p.strip() for p in peer_str.replace(';', ',').split(',') if p.strip()]
                         if peer_str else None)
        result = load_ticker(ticker, peer_override)
        if 'error' in result:
            exception_box.value = f'<span style="color:#c0392b;font-weight:600;">{result["error"]}</span>'
            return
        issuer = _state['issuer']
        issuer_label.value = (
            f'<div style="font-size:13px;color:#1a1f36;">'
            f'<b style="font-family:Consolas,monospace;font-size:15px;">{issuer["ticker"]}</b> '
            f'&middot; {issuer["name"]} '
            f'&middot; Rating: <b>{issuer["rating"]}</b> '
            f'&middot; Sector: <i>{issuer["sector"]}</i><br>'
            f'<span style="color:#888;font-size:11px;">Peer set ({len(_state["peer_tickers"])}): '
            f'{", ".join(_state["peer_tickers"])}</span>'
            f'</div>'
        )
        render()
    except Exception as e:
        exception_box.value = f'<span style="color:#c0392b;">Error: {e}</span>'
    finally:
        spinner.layout.visibility = 'hidden'

load_button.on_click(on_load_click)
for sl in (deal_size_slider, spread_shock_slider, rate_shock_slider, issuer_prem_slider, nic_slider):
    sl.observe(render, names='value')

# --- Layout ---

header = widgets.HTML(
    '<div style="background:#1a1f36;padding:14px 20px;border-radius:6px;margin-bottom:14px;">'
    '<div style="color:#fff;font-size:18px;font-weight:600;letter-spacing:0.3px;">'
    '<span style="color:#e8752a;">TIGRESS</span> Bond Pricing Engine '
    '<span style="background:#e8752a;color:#fff;font-size:10px;font-weight:700;padding:2px 8px;'
    'border-radius:3px;letter-spacing:0.5px;text-transform:uppercase;margin-left:8px;'
    'vertical-align:middle;">BQuant</span>'
    '</div>'
    '<div style="font-size:11px;color:#aab;margin-top:4px;letter-spacing:0.3px;">'
    'Live BQL &middot; Nelson-Siegel-Svensson &middot; Peer comparable analysis'
    '</div></div>'
)

ticker_row = widgets.VBox([
    widgets.HBox([ticker_input, peer_input, load_button, spinner]),
    issuer_label,
    exception_box,
], layout={'padding': '0 0 10px 0', 'border_bottom': '1px solid #eef0f3', 'margin': '0 0 14px 0'})

controls_block = widgets.VBox([
    widgets.HTML('<div style="color:#888;text-transform:uppercase;font-size:11px;letter-spacing:0.7px;'
                 'font-weight:700;border-bottom:1px solid #eef0f3;padding-bottom:6px;margin:8px 0 10px 0;">'
                 'Scenario Inputs</div>'),
    widgets.HBox([deal_size_slider, spread_shock_slider, rate_shock_slider]),
    widgets.HBox([issuer_prem_slider, nic_slider]),
    widgets.HBox([widgets.HTML('<span style="font-size:11px;color:#888;margin-right:8px;line-height:30px;">Presets:</span>'),
                  preset_base, preset_stress, preset_rally], layout={'margin': '6px 0 0 0'}),
    widgets.HTML('<div style="margin-top:14px;padding:10px 14px;background:#f0f4ff;border-left:3px solid #4a6cf7;'
                 'border-radius:0 6px 6px 0;font-size:11px;line-height:1.6;color:#334155;max-width:900px;">'
                 '<b>Methodology:</b> NSS spread curve fit to peer bonds (BQL <code>regression</code>), '
                 'plus credit, issuer, and NIC adjustments per tenor. '
                 'IPT = Final + 12 bps; Guidance = Final + 5 bps. '
                 'Big-deal bump: +1 bps &gt;$1.5B, +3 bps &gt;$2B.</div>'),
])

kpi_row = widgets.HBox([kpi_10y_spread, kpi_10y_yield, kpi_vs_base, kpi_peers],
                       layout={'margin': '0 0 14px 0'})

main_panel = widgets.VBox([
    kpi_row,
    ns_chart,
    ns_color_legend,
    widgets.HTML('<h4 style="margin:18px 0 8px 0;color:#1a1f36;">Indicative Pricing</h4>'),
    pricing_table,
    widgets.HBox([waterfall_chart, sens_chart], layout={'margin': '14px 0'}),
    widgets.HTML('<h4 style="margin:14px 0 8px 0;color:#1a1f36;">Coupon vs Current Yield</h4>'),
    widgets.HTML('<div style="font-size:11px;color:#888;margin-bottom:6px;">'
                 'Grey = coupon at issuance; orange = today\'s yield. Gap = repricing since issue.</div>'),
    coupon_vs_yield_chart,
    issuer_bonds_header,
    issuer_bonds_table,
    widgets.HTML('<h4 style="margin:14px 0 8px 0;color:#1a1f36;">Peer Comparable Bonds</h4>'),
    widgets.HTML('<div style="font-size:11px;color:#888;margin-bottom:6px;">'
                 'Currently outstanding peer bonds in the NSS fit. '
                 'Z-score: spread vs same-rating cohort (red = rich, green = cheap).</div>'),
    peer_table,
    widgets.HTML('<h4 style="margin:14px 0 8px 0;color:#1a1f36;">All Active USD Bonds (Issuer + Peers)</h4>'),
    widgets.HTML('<div style="font-size:11px;color:#888;margin-bottom:6px;">'
                 'Via <code>bondsuniv(active)</code> filtered by ticker. '
                 'Use these CUSIPs for downstream TRACE/ALLQ/IDOC lookups.</div>'),
    cusip_table,
    widgets.HTML('<h4 style="margin:14px 0 8px 0;color:#1a1f36;">Notes &amp; Flags</h4>'),
    notes_box,
])

dashboard = widgets.VBox([
    header,
    ticker_row,
    controls_block,
    widgets.HTML('<hr style="border:0;border-top:1px solid #eef0f3;margin:14px 0;">'),
    main_panel,
])

on_load_click()
dashboard
